<a href="https://colab.research.google.com/github/rlawhdqn-byte/data/blob/main/week05_5756299_%EA%B9%80%EC%A2%85%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [2]:
# 답안
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    # 생성자: 데이터(30개 샘플, 4개 특성)와 정답(30개) 텐서를 초기화합니다.
    def __init__(self):
        self.x_data = torch.randn(30, 4)
        self.y_data = torch.randint(0, 2, (30,))

    # 데이터셋 길이 반환: 전체 데이터의 개수(30개)를 반환합니다.
    def __len__(self):
        return len(self.x_data)

    # 인덱싱 연산: 요청한 인덱스(idx)에 해당하는 입력 데이터와 정답을 반환합니다.
    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# 작성한 Dataset 테스트
dataset = CustomDataset()
print("데이터셋 전체 개수:", len(dataset))
print("첫 번째 샘플(X, y) shape:", dataset[0][0].shape, dataset[0][1].shape)

데이터셋 전체 개수: 30
첫 번째 샘플(X, y) shape: torch.Size([4]) torch.Size([])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [3]:
# 답안
import torch
from torch.utils.data import Dataset, DataLoader

# 문제 1의 CustomDataset 정의
class CustomDataset(Dataset):
    def __init__(self):
        self.x_data = torch.randn(30, 4)
        self.y_data = torch.randint(0, 2, (30,))

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# Dataset 생성 및 DataLoader 연결 (batch_size=7)
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=7, shuffle=False)

# 1. 배치 개수 출력
print("Batch 개수:", len(dataloader))
print("-" * 45)

# 2. 각 배치별 입력/정답 shape 출력
for i, (x_batch, y_batch) in enumerate(dataloader):
    print(f"Batch {i+1} - 입력 shape: {x_batch.shape} | 정답 shape: {y_batch.shape}")

print("-" * 45)
# 3. 마지막 배치 크기가 다른 이유 출력
print("마지막 batch 크기가 다른 이유:")
print("전체 데이터 개수(30개)를 배치 크기(7)로 나누면 4개의 완벽한 배치가 만들어지고 2개가 남기 때문에 (30 = 7*4 + 2), 마지막 5번째 배치에는 남은 2개만 들어가게 됩니다.")

Batch 개수: 5
---------------------------------------------
Batch 1 - 입력 shape: torch.Size([7, 4]) | 정답 shape: torch.Size([7])
Batch 2 - 입력 shape: torch.Size([7, 4]) | 정답 shape: torch.Size([7])
Batch 3 - 입력 shape: torch.Size([7, 4]) | 정답 shape: torch.Size([7])
Batch 4 - 입력 shape: torch.Size([7, 4]) | 정답 shape: torch.Size([7])
Batch 5 - 입력 shape: torch.Size([2, 4]) | 정답 shape: torch.Size([2])
---------------------------------------------
마지막 batch 크기가 다른 이유:
전체 데이터 개수(30개)를 배치 크기(7)로 나누면 4개의 완벽한 배치가 만들어지고 2개가 남기 때문에 (30 = 7*4 + 2), 마지막 5번째 배치에는 남은 2개만 들어가게 됩니다.


## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


앞쪽 숫자 7 (batch_size): 한 번의 모델 연산(1개 배치)에 묶여서 들어가는 데이터 샘플의 개수입니다. (batch_size=7로 설정했으므로 한 묶음에 7개의 데이터가 포함됨)

뒤쪽 숫자 4 (특성 수): 각 데이터 샘플 1개가 가지고 있는 입력 변수(Feature)의 개수입니다. (Dataset 생성 시 정의한 4개의 특성값)

## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [4]:
# 답안
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 0. 문제 1의 Dataset 정의
class CustomDataset(Dataset):
    def __init__(self):
        # 학습이 정상적으로 진행(손실 감소)되도록 선형 관계를 가진 가상의 데이터 생성
        torch.manual_seed(42)
        self.x_data = torch.randn(30, 4)
        # y = 2*x1 - 3*x2 + x3 - 0.5*x4 + 노이즈 (0 또는 1 분류)
        target = (self.x_data @ torch.tensor([2.0, -3.0, 1.0, -0.5])) > 0
        self.y_data = target.float().unsqueeze(1) # shape: (30, 1)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# 데이터셋 및 데이터로더 생성
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=5, shuffle=True)

# 1. nn.Linear 모델, 손실 함수, 옵티마이저 생성
model = nn.Linear(in_features=4, out_features=1)
criterion = nn.BCEWithLogitsLoss() # 이진 분류용 손실 함수
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 2. 5단계 학습 루프 작성 (10 Epoch)
epochs = 10

print("--- 학습 시작 ---")
for epoch in range(1, epochs + 1):
    total_loss = 0.0

    for x_batch, y_batch in dataloader:
        # [단계 1: 예측] 모델에 입력을 전달하여 예측값(logit)을 계산
        pred = model(x_batch)

        # [단계 2: 손실] 예측값과 정답 간의 손실(Loss) 계산
        loss = criterion(pred, y_batch)

        # [단계 3: 초기화] 이전 단계에서 누적된 기울기(gradient) 초기화
        optimizer.zero_grad()

        # [단계 4: 역전파] 손실에 대한 모델 파라미터의 기울기 계산
        loss.backward()

        # [단계 5: 수정] 기울기를 바탕으로 모델 파라미터(가중치) 업데이트
        optimizer.step()

        total_loss += loss.item()

    # 3. Epoch마다 평균 손실 출력
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch:2d}/{epochs} | 평균 손실(Loss): {avg_loss:.4f}")

--- 학습 시작 ---
Epoch  1/10 | 평균 손실(Loss): 0.6964
Epoch  2/10 | 평균 손실(Loss): 0.5840
Epoch  3/10 | 평균 손실(Loss): 0.5043
Epoch  4/10 | 평균 손실(Loss): 0.4467
Epoch  5/10 | 평균 손실(Loss): 0.4086
Epoch  6/10 | 평균 손실(Loss): 0.3796
Epoch  7/10 | 평균 손실(Loss): 0.3555
Epoch  8/10 | 평균 손실(Loss): 0.3368
Epoch  9/10 | 평균 손실(Loss): 0.3208
Epoch 10/10 | 평균 손실(Loss): 0.3089
